# Milestone 2 — Two-site (and excited-state) DMRG for the market-making Hamiltonian

Follow-up to `market_making_tn_report.ipynb` (Milestone 1). This notebook implements report §12 milestone 2:

> *"Implement two-site DMRG and excited-state DMRG, including residual and quote metrics... reproduce Table 1 without forming the full state vector during optimization."*

**What Milestone 1 established:** exact diagonalization of the market-making Hamiltonian (sparse Lanczos) at $N=8$, followed by *post-hoc* TT-SVD compression of the exact ground state. That validates Proposition 5.2's MPO and reproduces Table 1 exactly, but it never tests whether an MPS *solver* can find the ground state without first computing it exactly — which is the actual point of the report's tensor-network approach at scale.

**What was already in the repo (and what wasn't):** `src/qubo_opt.jl` has a two-site sweep structure (`qubo_mps_dmrg_chain`) that inspired the sweep/environment bookkeeping style here, but it solves **diagonal QUBO/Ising chain energies via brute-force enumeration**, not a generic quantum Hamiltonian with off-diagonal hopping terms. `training.jl`'s Born-machine "DMRG-style" sweep has real two-site environments and SVD-based bond updates, but its local update is **gradient descent on a negative log-likelihood**, not an eigenvalue problem. Neither is a variational ground-state MPO eigensolver. This notebook's `src/dmrg.jl` is new: at each bond it forms the effective two-site Hamiltonian from **MPO environments** and diagonalizes it with **matrix-free Lanczos** (`KrylovKit.eigsolve`) — a genuine two-site DMRG.

All tensor contractions use `TensorOperations.@tensor` (not manual `reshape`/`permutedims`), to avoid the kind of silent index-ordering bug we had to work around in Milestone 1.

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using MPSFast
using LinearAlgebra
using SparseArrays
using Random
using Printf

  Activating 

project at `~/dev/Notes on Time Series Generation for Options Pricing/repos/Inter Science/MPSFast.jl`


[ Info: Precompiling MPSFast [c3f2d8a0-9e45-11ee-8c90-0242ac120002] (cache misses: wrong dep version loaded (4), mismatched flags (16))



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


## 1. The algorithm

For an MPO $\{W_i\}$ (report Prop. 5.2, bond dimension $K+2$) and an MPS $\{A_i\}$, define left/right environments
$$L_{j}[a',m,a] = \sum A_{1..j-1}\, W_{1..j-1}\, A_{1..j-1}, \qquad R_j[b',m,b] = \sum A_{j+1..N}\, W_{j+1..N}\, A_{j+1..N}.$$
At bond $(j,j+1)$, merge $A_j,A_{j+1}\to\psi[a,s_1,s_2,b]$ and diagonalize the **effective two-site Hamiltonian**
$$H^{\text{eff}}\psi = \sum L_j[a,m_0,a_2]\,W_j[m_0,s_1',s_1,m_1]\,W_{j+1}[m_1,s_2',s_2,m_2]\,R_{j+2}[b,m_2,b_2]\,\psi[a_2,s_1,s_2,b_2]$$
via matrix-free Lanczos (never forming $H^{\text{eff}}$ as a dense matrix, let alone the full Hamiltonian). The resulting $\psi$ is SVD-truncated back to bond dimension $\le\chi$, the canonical center moves to the next bond, and the environment on the vacated side is updated incrementally. Sweeping left→right→left repeatedly converges to the ground state.

For the **first excited state**, we fix a converged ground-state MPS $\psi_0$ and run the same sweep on the *deflated* operator $H + w\lvert\psi_0\rangle\langle\psi_0\rvert$ for $w$ larger than the spectral gap, using a second set of *overlap* environments between the trial state and $\psi_0$ (report §6.3 style).

Correctness of the residual $\lVert H\psi-E\psi\rVert$ and the energy $\langle\psi\vert H\vert\psi\rangle$ never requires the dense vector either: both are computed by applying the MPO to the MPS exactly (`apply_mpo_to_mps`, bond dims multiply by $K+2$) and taking MPS inner products.

## 2. Correctness: DMRG vs. exact diagonalization at small $N$

Before trusting DMRG at scale, we validate against Milestone 1's exact sparse-Lanczos ground truth.

**$N=2$** is the sharpest possible test: the "two-site block" *is* the entire system, so ground-state DMRG here reduces to a single local eigensolve with trivial (1×1×1) environments. It must match exact diagonalization to machine precision.

In [2]:
rng = MersenneTwister(1)
N2, Qs2 = 2, [1, 2]
Δ2 = [1.0, 1.0]
B2 = reshape([0.2, 0.2], 2, 1)
Σ2 = Diagonal(Δ2) + B2 * B2'
model2 = MarketMakingModel(N2, Qs2, Matrix(Σ2), [0.0, 0.0], 1.0, 1.0, [1.0, 1.0])
cores2 = build_mpo_cores(model2, Δ2, B2)
H2 = build_hamiltonian_sparse(model2)
vals2, vecs2 = exact_ground_states(H2; nev=2, rng=rng)

E2, mps2, _ = dmrg_ground_state(cores2; maxdim=16, n_sweeps=6, rng=MersenneTwister(2))
resid2 = dmrg_residual(mps2, cores2, E2)

@printf "exact E0        = %.12f\n" vals2[1]
@printf "DMRG  E0        = %.12f\n" E2
@printf "|ΔE|            = %.3e\n" abs(E2 - vals2[1])
@printf "Ritz residual   = %.3e\n" resid2

exact E0        = -2.484365568116
DMRG  E0        = -2.484365568116


|ΔE|            = 2.665e-15
Ritz residual   = 4.215e-08


**$N=5$**: exact diagonalization is still cheap here, so we can check that DMRG's ground- *and* excited-state energies converge to the exact values as $\chi$ grows, without ever using the exact answer inside the DMRG solver itself.

In [3]:
rng5 = MersenneTwister(4)
N5 = 5
Qs5 = fill(2, N5)
Δ5 = fill(0.8, N5)
B5 = 0.3 .* randn(rng5, N5, 2)
Σ5 = Diagonal(Δ5) + B5 * B5'
η5 = 0.9 .+ 0.2 .* rand(rng5, N5)
model5 = MarketMakingModel(N5, Qs5, Matrix(Σ5), zeros(N5), 1.0, 1.1, η5)
cores5 = build_mpo_cores(model5, Δ5, B5)
H5 = build_hamiltonian_sparse(model5)
vals5, vecs5 = exact_ground_states(H5; nev=2, rng=rng5)
E0_exact5, E1_exact5 = vals5

@printf "exact: E0=%.8f  E1=%.8f  gap=%.8f\n\n" E0_exact5 E1_exact5 (E1_exact5 - E0_exact5)
@printf "%-4s %-16s %-14s %-14s\n" "χ" "DMRG E0" "|ΔE0|" "residual"
for χ in (4, 8, 16)
    E, mps, _ = dmrg_ground_state(cores5; maxdim=χ, n_sweeps=8, rng=MersenneTwister(10 + χ))
    resid = dmrg_residual(mps, cores5, E)
    @printf "%-4d %-16.10f %-14.3e %-14.3e\n" χ E abs(E - E0_exact5) resid
end

exact: E0=-6.48500831  E1=-5.29043647  gap=1.19457184

χ    DMRG E0          |ΔE0|          residual      


4    -6.4850004810    7.828e-06      6.435e-03     
8    -6.4850083027    6.518e-09      2.260e-04     


16   -6.4850083093    7.994e-14      1.066e-06     


In [4]:
E0_5, gs_mps5, _ = dmrg_ground_state(cores5; maxdim=16, n_sweeps=10, rng=MersenneTwister(99))
E1_5, es_mps5, _ = dmrg_excited_state(cores5, gs_mps5; maxdim=16, n_sweeps=10, rng=MersenneTwister(100))

@printf "ground : DMRG=%.10f  exact=%.10f  |Δ|=%.3e\n" E0_5 E0_exact5 abs(E0_5 - E0_exact5)
@printf "excited: DMRG=%.10f  exact=%.10f  |Δ|=%.3e\n" E1_5 E1_exact5 abs(E1_5 - E1_exact5)

ground : DMRG=-6.4850083093  exact=-6.4850083093  |Δ|=8.704e-14
excited: DMRG=-5.2904364674  exact=-5.2904364675  |Δ|=1.380e-10


Both the ground- and excited-state solvers converge to the exact spectrum as $\chi$ grows, without exact diagonalization anywhere in the loop. This is the core correctness evidence for the DMRG implementation.

## 3. Reproducing report Table 1 (§8) at $N=8$ — without ever forming the state vector

Same 8-asset model as Milestone 1. This time, at each $\chi$, `dmrg_ground_state` is run from a **random initial MPS** — the $390{,}625$-dimensional exact ground state (`ϕ0_exact` below) is used *only afterwards*, to measure how close the purely-variational answer is. It plays no role in the solver.

In [5]:
N = 8
Qs = fill(2, N)
k, γ = 1.0, 1.1
μ = zeros(N)
Δ = [0.70, 0.75, 0.80, 0.85, 0.85, 0.80, 0.75, 0.70]
B = [0.30 0.28; 0.34 0.24; 0.38 0.20; 0.42 0.16; 0.46 -0.16; 0.50 -0.20; 0.54 -0.24; 0.58 -0.28]
Σ = Diagonal(Δ) + B * B'
η = [1.00, 0.95, 1.05, 0.90, 1.10, 1.00, 0.92, 1.08]
model = MarketMakingModel(N, Qs, Matrix(Σ), μ, k, γ, η)
cores = build_mpo_cores(model, Δ, Matrix(B))

H = build_hamiltonian_sparse(model)
vals, vecs = exact_ground_states(H; nev=2, rng=MersenneTwister(42))
E0_exact, E1_exact = vals
ϕ0_exact = vecs[1]
@printf "(reference, milestone 1) exact E0 = %.8f, E1 = %.8f, gap = %.8f\n" E0_exact E1_exact (E1_exact - E0_exact)

dims = site_dims(model)
edges = central_grid_edges(N)
println("Hilbert space dimension: ", prod(dims), " (used only for the reference computation above)")

(reference, milestone 1) exact E0 = -10.36159622, E1 = -9.20407484, gap = 1.15752138
Hilbert space dimension: 

390625 (used only for the reference computation above)


In [6]:
@printf "%-4s %-16s %-14s %-14s %-16s %-10s\n" "χ" "DMRG E0" "|ΔE0|" "residual" "core rmse" "time (s)"
dmrg_rows = NamedTuple[]
for χ in (4, 8, 16, 32)
    t0 = time()
    E, mps, hist = dmrg_ground_state(cores; maxdim=χ, n_sweeps=12, rng=MersenneTwister(1000 + χ), cutoff=1e-14)
    elapsed = time() - t0
    resid = dmrg_residual(mps, cores, E)
    ϕ_dmrg = mps_to_lex_vector([Array(a) for a in mps], dims)
    ϕ_dmrg = align_and_normalize(ϕ_dmrg, ϕ0_exact)
    rmse, maxerr = quote_log_ratio_errors(ϕ_dmrg, ϕ0_exact, Qs, edges)
    push!(dmrg_rows, (χ=χ, E=E, dE=E - E0_exact, resid=resid, rmse=rmse, maxerr=maxerr, time=elapsed))
    @printf "%-4d %-16.10f %-14.3e %-14.3e %-16.3e %-10.1f\n" χ E abs(E - E0_exact) resid rmse elapsed
end

χ    DMRG E0          |ΔE0|          residual       core rmse        time (s)  
4    -10.3615471878   4.903e-05      2.158e-02      6.506e-03        0.3       


8    -10.3615960980   1.208e-07      1.137e-03      2.520e-04        0.8       
16   -10.3615962186   1.394e-10      4.066e-05      9.307e-06        1.5       


32   -10.3615962188   1.581e-13      1.262e-06      1.539e-07        3.4       


Compare to Milestone 1's Table 1 reproduction, where the *same* $\chi$'s were obtained by TT-SVD-compressing the exact ground state post-hoc:

| χ | Table 1 (post-hoc TT-SVD of exact ψ) | Here (pure DMRG, no exact diag) |
|---|---|---|
| 4  | rmse ≈ 7.48e-3 | see `dmrg_rows` above |
| 8  | rmse ≈ 2.70e-4 | " |
| 16 | rmse ≈ 1.04e-5 | " |
| 32 | rmse ≈ 1.61e-7 | " |

The two columns land on the same order of magnitude at every $\chi$ — variational two-site DMRG, which never touches the exact answer, reaches essentially the same accuracy as compressing the exact answer after the fact. This directly answers the report's central open question (§11.1): **for this factor-covariance MPO, low MPO rank does translate into a solver that converges to low MPS rank, without needing exact diagonalization first.**

## 4. Excited state at $N=8$

In [7]:
E0_dmrg8, gs_mps8, _ = dmrg_ground_state(cores; maxdim=32, n_sweeps=12, rng=MersenneTwister(5000))
E1_dmrg8, es_mps8, _ = dmrg_excited_state(cores, gs_mps8; maxdim=32, n_sweeps=12, rng=MersenneTwister(5001))

@printf "ground : DMRG=%.8f  exact=%.8f  |Δ|=%.3e\n" E0_dmrg8 E0_exact abs(E0_dmrg8 - E0_exact)
@printf "excited: DMRG=%.8f  exact=%.8f  |Δ|=%.3e\n" E1_dmrg8 E1_exact abs(E1_dmrg8 - E1_exact)
@printf "gap    : DMRG=%.8f  exact=%.8f\n" (E1_dmrg8 - E0_dmrg8) (E1_exact - E0_exact)

ground : DMRG=-10.36159622  exact=-10.36159622  |Δ|=1.634e-13
excited: DMRG=-9.20407484  exact=-9.20407484  |Δ|=2.085e-11


gap    : DMRG=1.15752138  exact=1.15752138


## 5. Genuine scale: $N=20$, Hilbert dimension ≈ 3.5 billion

This is the actual point of milestone 2: a system where exact diagonalization (Milestone 1's approach) is simply impossible — there is no dense or sparse vector of this size to ever form — but two-site DMRG doesn't care, because it only ever touches $O(\chi^2 d^2 R)$-sized local tensors.

In [8]:
N20 = 20
Qs20 = fill(1, N20)  # d = 2Q+1 = 3 per site
rng20 = MersenneTwister(7)
Δ20 = 0.6 .+ 0.3 .* rand(rng20, N20)
B20 = 0.25 .* randn(rng20, N20, 3)
Σ20 = Diagonal(Δ20) + B20 * B20'
η20 = 0.9 .+ 0.2 .* rand(rng20, N20)
model20 = MarketMakingModel(N20, Qs20, Matrix(Σ20), zeros(N20), 1.0, 1.1, η20)
cores20 = build_mpo_cores(model20, Δ20, Matrix(B20))

dim20 = prod(site_dims(model20))
@printf "N=%d, per-site dim=%d, full Hilbert dimension = %.3e (never formed)\n\n" N20 site_dims(model20)[1] Float64(dim20)

@printf "%-4s %-16s %-14s %-10s\n" "χ" "DMRG E0" "residual" "time (s)"
for χ in (8, 16, 32, 64)
    t0 = time()
    E, mps, _ = dmrg_ground_state(cores20; maxdim=χ, n_sweeps=10, rng=MersenneTwister(200 + χ))
    elapsed = time() - t0
    resid = dmrg_residual(mps, cores20, E)
    @printf "%-4d %-16.8f %-14.3e %-10.1f\n" χ E resid elapsed
end

N=20, per-site dim=3, full Hilbert dimension = 3.487e+09 (never formed)

χ    DMRG E0          residual       time (s)  


8    -23.41109040     3.466e-02      0.6       
16   -23.41124937     3.947e-03      2.1       


32   -23.41125082     2.377e-04      5.8       
64   -23.41125083     1.691e-05      19.0      


Energy converges monotonically and the Ritz residual drops steadily as $\chi$ grows — the hallmark of a controlled variational calculation — on a system roughly $9{,}000\times$ larger than what exact diagonalization reached in Milestone 1.

## What this establishes, and what's next

- **New code, not reused code.** Despite the initial guess that a suitable two-site DMRG already existed in the repo, neither `qubo_opt.jl`'s chain-QUBO solver nor `training.jl`'s NLL-gradient sweep implements a variational MPO eigensolver. `src/dmrg.jl` is a from-scratch two-site (and excited-state) DMRG built on MPO environments + matrix-free Lanczos, validated to match exact diagonalization to machine precision at $N=2$ and to the exact spectrum as $\chi\to$ generous at $N=5$.
- **Milestone 2 (§12) is complete**: ground- and excited-state two-site DMRG, residual (`dmrg_residual`) and quote-ratio (`quote_log_ratio_errors`) metrics, and a from-scratch reproduction of Table 1's accuracy at $N=8$ *without* forming the $390{,}625$-dimensional vector during optimization.
- **New evidence beyond the report**: DMRG converges cleanly at $N=20$ (Hilbert dimension $\approx 3.5\times10^9$), which is squarely in the regime the report's tensor-network approach is *for* and which exact diagonalization cannot touch.
- **A practical fix worth noting**: the first version of `apply_Heff_twosite` used a single 5-operand `@tensor` expression and let `TensorOperations` pick the contraction order; at $\chi=32$ this picked a bad order and didn't finish in 30+ minutes. Splitting it into four explicit pairwise contractions (one MPO layer/environment at a time — the standard DMRG contraction order) brought $\chi=32$ down to under 4 seconds, with identical numerical results. Worth remembering for milestone 3.
- **§10.1 progress (see `notes/coutinho/market_making_model_notes.tex`, §6):** exact-benchmark family mostly complete; scaling started at $N=20$; finite horizon, Doob chain, and remaining baselines still open.
- **Open next steps** (report §12 / §11.8): full phase-diagram scan of $\chi_\epsilon$ vs. $(N,K,\text{factor strength})$; finite-horizon MPS evolution; Doob-chain / mixing-time check; Gaussian and factor-grid baselines; quantum-computing extension (QITE/VQE).